# SRΨ-Engine Ablation Study - Colab Edition

## 🚀 快速开始（3步）

**⚠️ 重要：先启用 GPU！**
1. 点击菜单：**Runtime** → **Change runtime type**
2. Hardware accelerator 选择：**T4 GPU** 或 **A100 GPU**
3. 点击 **Save**
4. 重新运行下面这个 cell 检查 GPU

---

## Step 1: 检查 GPU

In [ ]:
import torch
import os

print('=== GPU Status Check ===')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('\n🎉 GPU is ready! You can proceed to Step 2.')
else:
    print('\n❌ No GPU detected!')
    print('\n🔧 Please enable GPU:')
    print('   1. Click: Runtime → Change runtime type')
    print('   2. Hardware accelerator: T4 GPU')
    print('   3. Click Save')
    print('   4. Run this cell again')

## Step 2: 克隆仓库（使用 HTTPS，无需认证）

In [ ]:
# 使用 HTTPS 克隆（不需要认证）
!git clone https://github.com/Mozilla2004/srpsi-engine-tiny.git

# 进入项目目录
%cd srpsi-engine-tiny

# 验证克隆成功
print('\n=== Repository cloned successfully ===')
!ls -la
print('\n✓ Ready to proceed!')

## Step 3: 安装依赖

In [ ]:
# 安装必需的包
!pip install -q pyyaml tqdm

import yaml
from tqdm import tqdm

print('\n=== Dependencies installed ===')
print('✓ pyyaml')
print('✓ tqdm')
print('✓ torch')
print('✓ numpy')

## Step 4: 生成/检查训练数据

In [ ]:
# 检查数据文件
import os

data_file = 'data/burgers_1d.npy'

if os.path.exists(data_file):
    size_mb = os.path.getsize(data_file) / (1024 * 1024)
    print(f'\n=== Data already exists ===')
    print(f'File: {data_file}')
    print(f'Size: {size_mb:.1f} MB')
    print('✓ Ready to train!')
else:
    print('\n=== Generating training data ===')
    print('This may take a few minutes...')
    !python src/data_gen.py
    
    if os.path.exists(data_file):
        size_mb = os.path.getsize(data_file) / (1024 * 1024)
        print(f'\n✓ Data generated: {size_mb:.1f} MB')
    else:
        print('✗ Data generation failed!')

## Step 5: 运行实验

**选择要运行的实验**：

可用选项：
- `srpsi_real`: SRΨ Real-only (Exp2)
- `srpsi_no_r`: SRΨ without Rhythm operator (Exp3)
- `conv_baseline`: Pure Convolution baseline (Exp4)
- `transformer_rel_pe`: Transformer with Relative Position (Exp5)

**提示**：打开多个标签页可以并行运行多个实验！

In [ ]:
# ============================================================
# ⚙️ 配置您的实验
# ============================================================

model_name = 'srpsi_real'  # 修改这里选择实验
num_epochs = 80            # 训练轮数（默认 80）

# ============================================================

print(f'\n=== Starting Experiment ===')
print(f'Model: {model_name}')
print(f'Epochs: {num_epochs}')
print(f'Output: outputs/ablation_{model_name}')
print(f'Estimated time: ~30-45 minutes (on GPU)')
print('='*60)

# 运行训练
!python src/train.py \
  --config config/burgers.yaml \
  --model {model_name} \
  --data data/burgers_1d.npy \
  --output outputs/ablation_{model_name} \
  --epochs {num_epochs}

## Step 6: 检查训练结果

In [ ]:
# 查找并显示最佳 checkpoint
from glob import glob

checkpoint_dir = f'outputs/ablation_{model_name}/{model_name}/checkpoints'

if os.path.exists(checkpoint_dir):
    checkpoints = glob(f'{checkpoint_dir}/best*.pt')
    
    if checkpoints:
        best_model = checkpoints[0]
        size_mb = os.path.getsize(best_model) / (1024 * 1024)
        print(f'\n=== Training Complete ===')
        print(f'✅ Best model: {best_model}')
        print(f'   Size: {size_mb:.2f} MB')
        
        final_models = glob(f'{checkpoint_dir}/final*.pt')
        if final_models:
            print(f'✅ Final model: {final_models[0]}')
    else:
        print('\n⚠️ No best checkpoint found.')
        print('Training may still be in progress.')
        print('\nAll checkpoints:')
        !ls -lh {checkpoint_dir}/
else:
    print(f'\n✗ Checkpoint directory not found: {checkpoint_dir}')

## Step 7: 下载结果

### 方法 1：通过 Colab 界面下载
1. 点击左侧 📁 文件图标
2. 导航到：`srpsi-engine-tiny/outputs/ablation_{model_name}/{model_name}/checkpoints/`
3. 右键点击 `best*.pt` → Download

### 方法 2：通过代码下载

In [ ]:
from google.colab import files

checkpoint_path = f'outputs/ablation_{model_name}/{model_name}/checkpoints/best.pt'

if os.path.exists(checkpoint_path):
    print(f'Downloading: {checkpoint_path}')
    files.download(checkpoint_path)
else:
    print(f'File not found: {checkpoint_path}')
    print('\nAvailable files:')
    !ls -lh outputs/ablation_{model_name}/{model_name}/checkpoints/ 2>/dev/null || echo 'Directory not found'

## 📊 快速参考

### 并行运行多个实验

要在 ~40 分钟内完成所有 5 个实验：

1. **打开 4 个浏览器标签页**，每个都打开这个 notebook

2. **在每个标签页中设置不同的 `model_name`**：
   - 标签 1: `model_name = 'srpsi_real'`
   - 标签 2: `model_name = 'srpsi_no_r'`
   - 标签 3: `model_name = 'conv_baseline'`
   - 标签 4: `model_name = 'transformer_rel_pe'`

3. **在每个标签页运行 Step 5**

4. **所有实验将同时进行，40 分钟后全部完成！**

### 训练时间预估（GPU）

| 实验 | 每个Epoch | 总时间（80 epochs） |
|------|----------|-------------------|
| SRΨ Real | ~20秒 | ~30分钟 |
| SRΨ w/o R | ~25秒 | ~35分钟 |
| Conv Baseline | ~15秒 | ~25分钟 |
| Transformer | ~30秒 | ~45分钟 |

### 故障排除

**问题**: GPU available: False
- **解决**: Runtime → Change runtime type → GPU → Save → 重新运行 Step 1

**问题**: Git clone 失败
- **解决**: 确认仓库是 public 的，或使用上面的 HTTPS URL

**问题**: Out of memory
- **解决**: Runtime → Restart runtime，然后重新运行

**问题**: Colab 自动休眠
- **解决**: 查看下一个 cell 的防休眠代码

## 💡 防止 Colab 休眠（可选）

In [ ]:
# 运行这个 cell 可以防止 Colab 在长时间训练时休眠
# (在训练过程中，这个脚本会每分钟点击一次连接按钮)

%%javascript
function ClickConnect(){
    console.log("Working...");
    document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)

print('✅ Anti-sleep script activated!')
print('   (This will keep Colab awake during long training runs)')

## 📖 完整使用流程

### 第一次使用（初始设置）

1. **启用 GPU**: Runtime → Change runtime type → GPU → Save
2. **运行 Step 1**: 检查 GPU 是否可用
3. **运行 Step 2**: 克隆仓库
4. **运行 Step 3**: 安装依赖
5. **运行 Step 4**: 生成训练数据（约 2-3 分钟）

### 运行实验

6. **修改 Step 5** 中的 `model_name` 为您想运行的实验
7. **运行 Step 5**: 开始训练（约 30-45 分钟）
8. **运行 Step 6**: 检查结果
9. **运行 Step 7**: 下载模型

### 运行多个实验（并行）

重复步骤 6-9，但在不同的浏览器标签页中使用不同的 `model_name`

---

**祝训练顺利！如有问题，请查看上面的故障排除部分。** 🚀